# Dokumentembeddings med borealis-embed og sentence-transformers-biblioteket

Vi lager dokumentvektorer for hver tekst i subsettet med
[NbAiLab/borealis-embed-212m](https://huggingface.co/NbAiLab/borealis-embed-212m) via
sentence-transformers.  
Embeddingene blir flatet ned og visualisert med den samme koden som BoW-embeddingene
(`visualize.py`).

## Les inn subsettet

In [ ]:
from pathlib import Path

import pandas as pd

SUBSET_DIR = Path("data/maalfrid/subset")
paths = sorted(SUBSET_DIR.glob("*/*.txt"))
texts = pd.Series(
    [path.read_text(encoding="utf-8") for path in paths],
    index=pd.Index([f"{path.parent.name}/{path.stem}" for path in paths]),
)
len(texts)

## Lag embeddinger

Tekster som er lengre enn modellens maksimale sekvenslengde blir kuttet.  

In [ ]:
import gc

import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = "NbAiLab/borealis-embed-212m"
model = SentenceTransformer(MODEL_NAME)
print(f"Max sequence length: {model.max_seq_length}")

embeddings = model.encode(texts.tolist(), batch_size=4, show_progress_bar=True)

# del model
# gc.collect()
# torch.cuda.empty_cache()

embeddings.shape

## Eksempel på en embedding

In [ ]:
embeddings[0]

## Flat ned til 2D-vektorer for visualisering

In [ ]:
from visualize import plot_by_domain, project_2d

projection = project_2d(embeddings, texts.index)
projection.head()

## Scatterplot etter kildedomene

Hvert domene har sin egen farge og form.  
Klikk på et domene i forklaringen for å skjule eller vise det, og dobbeltklikk for å vise bare det domenet.  
Hold over en prikk for å se domenet og hele teksten i panelet til høyre. Klikk på prikken for å feste teksten, så du kan scrolle i den.

In [ ]:
plot_by_domain(projection, texts, "t-SNE of Borealis embeddings")

## Mest og minst like domener

Hvert domene representeres ved sentroiden sin (gjennomsnittet av dokumentembeddingene).  
Domenene sammenlignes med cosinuslikhet mellom sentroidene.

In [ ]:
from embedding_similarity import (
    centroid_similarities,
    compute_centroids,
    most_and_least_similar,
)

centroids = compute_centroids(embeddings, projection["domain"])
similarities = centroid_similarities(centroids)
most_and_least_similar(similarities).round(3)